# Prepare NylC Diversity Panel sequences for AlphaFold2/ColabFold modeling

Reads the NylC-GYAQ + homolog sequences and the multiple sequence alignment (MSA), identifies the autocatalytic cleavage site
(Thr267 in NylC) in every sequence, and writes ColabFold-ready FASTA files (alpha:beta chains, n-mer of heterodimers) plus
`cleavage_site.csv`. The same file is provided in `data/` and is what the post-processing notebook reads, so running this notebook is optional.

The panel contains NylC-GYAQ, the original diversity panel (nyl001-nyl095; models on Zenodo, record 14976747) and 189 additional
homologs (dp001-dp189). The 24 extra picks (nyl096-nyl119) are not included.

Nothing is written to `data/`; all outputs go to `RESULTS_DIR`.

In [ ]:
import os, sys, glob, json, math, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

# ---- Paths (override with environment variables) -------------------------------------
DATA_DIR    = Path(os.environ.get('NYLC_DATA_DIR', 'data'))              # inputs (read-only)
RESULTS_DIR = Path(os.environ.get('NYLC_RESULTS_DIR', 'results'))        # everything written by this notebook
LEGACY_DIR  = Path(os.environ.get('NYLC_LEGACY_DIR', 'legacy'))          # legacy outputs, for consistency checks
N_JOBS      = int(os.environ.get('NYLC_N_JOBS', '8'))
FORCE       = os.environ.get('NYLC_FORCE', '0') == '1'                   # recompute even if outputs exist
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('data:', DATA_DIR.resolve(), '\nresults:', RESULTS_DIR.resolve())
NMER = 4                     # 1 = monomer, 2 = dimer, 4 = tetramer of heterodimers
RUN_MAFFT = False            # True: build the MSA here with MAFFT L-INS-i instead of using the provided MSA
MSA_FILE = DATA_DIR / 'nylc_all.mafft.fasta'
FASTA_DIR = RESULTS_DIR / 'fasta_af2'
FASTA_DIR.mkdir(exist_ok=True)

In [ ]:
def parse_fasta(filename):
    """Return (headers, sequences). Handles blank lines; sequences are upper-cased."""
    header, sequence = [], []
    with open(filename) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                header.append(line[1:]); sequence.append([])
            else:
                sequence[-1].append(line.upper())
    return header, [''.join(s) for s in sequence]

## Unaligned sequences (optional MSA step)
The official MSA was made with the MAFFT extension in Jalview (L-INS-i preset). It is provided in `data/nylc_all.mafft.fasta` and is used by default,
so that MSA-derived features remain identical to the legacy dataset. Set `RUN_MAFFT = True` to regenerate it (requires `mafft`; results may differ slightly).

In [ ]:
seqs_df = pd.read_csv(DATA_DIR / 'nylc_all_seqs.csv')
# NylC-GYAQ must come first: it is the reference for numbering.
seqs_df = pd.concat([seqs_df[seqs_df.enzyme == 'nylc-gyaq'], seqs_df[seqs_df.enzyme != 'nylc-gyaq'].sort_values('enzyme')])
unaligned = RESULTS_DIR / 'nylc_all_combined.fasta'
with open(unaligned, 'w') as fh:
    for enz, seq in zip(seqs_df.enzyme, seqs_df.AAseq):
        fh.write(f'>{enz}\n' + '\n'.join(seq[i:i+72] for i in range(0, len(seq), 72)) + '\n')
print(len(seqs_df), 'sequences written to', unaligned)

if RUN_MAFFT:
    MSA_FILE = RESULTS_DIR / 'nylc_all.mafft.generated.fasta'
    with open(MSA_FILE, 'w') as out:
        subprocess.run(['mafft', '--localpair', '--maxiterate', '1000', '--thread', str(N_JOBS), str(unaligned)], stdout=out, check=True)
print('Using MSA:', MSA_FILE)

## Write model-ready FASTA files
NylC autocatalytically cleaves into an alpha and a beta chain; the beta chain N-terminus is the catalytic Thr267 (motif `TTIS` in NylC-GYAQ).
The cleavage column is located in the NylC-GYAQ row of the MSA and every sequence is split at that column.

In [ ]:
def write_nx2_fastas(nmer, msa_file, fasta_dir, csv_path):
    names, seqs = parse_fasta(msa_file)
    short = [n.partition('/')[0].lower() for n in names]
    assert short[0] == 'nylc-gyaq', f'first MSA sequence must be nylc-gyaq, got {short[0]}'
    assert len(set(short)) == len(short), 'duplicate sequence names in MSA'
    col = seqs[0].find('TTIS')               # alignment column of Thr267 (first residue of beta chain)
    assert col != -1 and '-' not in seqs[0][col:col+4], "'TTIS' motif not found contiguously in NylC-GYAQ"
    data = []
    for name, seq in zip(short, seqs):
        alpha = seq[:col].replace('-', '')
        beta = seq[col:].replace('-', '')
        data.append({'Enzyme': name, 'Thr267': len(alpha) + 1})
        with open(Path(fasta_dir) / f'{name}_{nmer}x2.fasta', 'w') as fh:
            fh.write(f'>{name}\n' + ':'.join([alpha, beta] * nmer) + '\n')
    pd.DataFrame(data).to_csv(csv_path, index=False)
    return pd.DataFrame(data)

cleavage = write_nx2_fastas(NMER, MSA_FILE, FASTA_DIR, RESULTS_DIR / 'cleavage_site.csv')
print(len(cleavage), 'FASTA files written to', FASTA_DIR); cleavage.head()

## Consistency check against legacy output
Compares the new `cleavage_site.csv` with the provided `data/cleavage_site.csv`, and the ColabFold FASTA files with those used to generate the published models.

In [ ]:
leg = pd.read_csv(DATA_DIR / 'cleavage_site.csv')
m = cleavage.merge(leg, on='Enzyme', suffixes=('_new', '_legacy'), how='left')
print('enzymes in new but not legacy:', m.Thr267_legacy.isna().sum())
print('cleavage-site mismatches:', int((m.Thr267_new != m.Thr267_legacy).sum()))
legacy_fa = Path(os.environ.get('NYLC_LEGACY_FASTA_DIR', ''))
if legacy_fa.is_dir():
    import re
    def legacy_file(n):   # the legacy FASTA directory also uses two-digit names for nyl001-nyl099 (nyl01_4x2.fasta)
        for cand in (n, re.sub(r'^nyl0(\d\d)$', r'nyl\1', n)):
            if (legacy_fa / f'{cand}_{NMER}x2.fasta').exists():
                return legacy_fa / f'{cand}_{NMER}x2.fasta'
    pairs = {n: legacy_file(n) for n in cleavage.Enzyme}
    nofile = [n for n, f in pairs.items() if f is None]
    bad = [n for n, f in pairs.items() if f is not None and f.read_text().split('\n', 1)[1] != (FASTA_DIR / f'{n}_{NMER}x2.fasta').read_text().split('\n', 1)[1]]
    print('no legacy FASTA found for:', nofile)
    print('FASTA sequences differing from legacy:', bad)
else:
    print('Set NYLC_LEGACY_FASTA_DIR to compare the FASTA files with the legacy ones.')